# Counter-Trend

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None) # Show all columns when printing a dataframe

## Prepare data

In [ ]:
df = pd.read_excel('/Users/jlaw/projects/stern/systematic-investing/data/COUNTER_TREND_DATA.xlsx', header=1)
df = df.iloc[2:] # Remove rows where Hedge hasn't computed yet
df['Date'] = pd.to_datetime(df['#Date'], format='%Y%m%d')
df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Roll', 'DayRange', 'Av20R', 'PrvHiWRoll', 'HitLevel', 'Hit?', 'HitAt', 'ExitPr', 'Ret2C', 'BuyNhold']]
df.head()

## Setup data

In [ ]:
df['DayRange'] = df['High'] - df['Low']
df['Av20R'] = df['DayRange'].rolling(window=20).mean()
df['PrvHiWRoll'] = df['High'].shift(1) + df['Roll']
# df.head(51)

## Create strategy

In [ ]:
P = 2.2
df['HitLevel'] = df['PrvHiWRoll'] - (P * df['Av20R'])
df['Hit?'] = df['Low'] < df['HitLevel']
# HitAt is the HitLevel unless the Open is below the HitLevel, in which case it's the Open
df['HitAt'] = np.where(df['Open'] < df['HitLevel'], df['Open'], df['HitLevel'])
df['HitAt'] = np.where(df['Hit?'], df['HitAt'], 0)
df['ExitPr'] = df['Close']
df['Ret2C'] = np.where(df['Hit?'], (df['ExitPr'] - df['HitAt']) / df['HitAt'], 0) # Compare zero vs NaN for diff Sharpe
df['BuyNhold'] = (df['Close'] - df['Open']) / df['Open']
df.head(100)

## Evaluate strategy

In [ ]:
df[['strategy_cumulative_ret', 'buy_and_hold_cumulative_ret']] = (1 + df[['Ret2C', 'BuyNhold']]).cumprod() - 1
df[['strategy_cumulative_ret', 'buy_and_hold_cumulative_ret']].plot();

### Compare sharpes

In [ ]:
print('Strategy sharpe', ((df['Ret2C'].mean() / df['Ret2C'].std()) * np.sqrt(252)).round(2))
print('Buy and Hold sharpe', ((df['BuyNhold'].mean() / df['BuyNhold'].std() * np.sqrt(252))).round(2))

In [ ]:
# Plot raw returns of Strategy vs Buy and Hold
df[['Ret2C', 'BuyNhold']].plot.scatter(x='Ret2C', y='BuyNhold');

In [ ]:
# Count of returns per quadrant
# Two dataframes so I can sort them separately and qcut into quadrants
df.dropna()
df_strategy = df[['Ret2C']].copy()
df_buy_and_hold = df[['BuyNhold']].copy()
df_strategy = df_strategy.sort_values(by='Ret2C').reset_index(drop=True)
df_buy_and_hold = df_buy_and_hold.sort_values(by='BuyNhold').reset_index(drop=True)
df_strategy['quadrant'] = pd.qcut(df_strategy.index, q=4, labels=False)
df_buy_and_hold['quadrant'] = pd.qcut(df_buy_and_hold.index, q=4, labels=False)
# Merge the quadrants back together
df_quadrants = df_strategy.merge(df_buy_and_hold, left_index=True, right_index=True)
# Count the number of returns in each quadrant
quadrant_counts = df_quadrants.groupby(['quadrant_x', 'quadrant_y']).size().unstack(fill_value=0)
print(quadrant_counts)


In [ ]:
#cant figure out how to group the returns out of time